In [30]:
from langchain_community.document_loaders import PyMuPDFLoader, UnstructuredPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
# import faiss
import os

In [31]:
# def load_pdf_from_folder(folder_path: str):
#     documents = []
#     for file in os.listdir(folder_path):
#         if file.endswith(".pdf"):
#             file_path = os.path.join(folder_path, file)
#             try:
#                 loader = PyMuPDFLoader(file_path)
#                 docs = loader.load()
                
#                 if not any(doc.page_content.strip() for doc in docs):
#                     raise ValueError("Empty text, fallback to OCR")

#             except Exception:
#                 print(f"couldent load {file}")
#             documents.extend(docs)
#     return documents

In [32]:
# docs = load_pdf_from_folder("C:\\Users\\Santosh\\Desktop\\AIC\\doc")
pdf_path = r"C:\Users\Santosh\Desktop\AIC\doc\AIC rag doc.pdf"

loader = PyMuPDFLoader(pdf_path)

docs = loader.load()

In [33]:
print(docs)

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': '', 'creationdate': '2026-09-02T20:09:02+05:30', 'source': 'C:\\Users\\Santosh\\Desktop\\AIC\\doc\\AIC rag doc.pdf', 'file_path': 'C:\\Users\\Santosh\\Desktop\\AIC\\doc\\AIC rag doc.pdf', 'total_pages': 30, 'format': 'PDF 1.7', 'title': 'Microsoft Word - AIC_Agent3_Ecommerce_Solution_Playbook', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-09-02T20:09:02+05:30', 'trapped': '', 'modDate': "D:20260902200902+05'30'", 'creationDate': "D:20260902200902+05'30'", 'page': 0}, page_content='AIC | AI Agent 3 — E-commerce Solution Playbook \nUse recommendations only after Agent 2 has identified the anomaly and supported cause. Validate high-cost actions with experiments. \nE-COMMERCE \nSOLUTION PLAYBOOK \nRAG Knowledge Base for AI Agent 3 \nA 30-page anomaly-to-action intervention handbook designed for solution generation in the AIC e-commerce \nanalytics architecture. \nAI AGENT 1  →  ANOMALY     |     AI AGENT

In [34]:
def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size, chunk_overlap = chunk_overlap, separators= ["\n\n", "\n", " ", ""] )
    split = text_splitter.split_documents(documents=documents)
    return split

In [35]:
chunks = split_documents(documents=docs)

In [36]:
print(len(chunks))

112


In [37]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",model_kwargs={"device": "cuda"})
# vectordb = Chroma.from_documents(documents=chunks, embedding= embedding_model,persist_directory="./chroma_db")
# vectordb = Chroma(persist_directory="./chroma_db", embedding_function=embedding_model)
# batch_size =64
# for i in range(0, len(chunks),batch_size):
#     batch = chunks[i:i+batch_size]
#     try:
#         vectordb.add_documents(batch)
#     except Exception as e:
#         print(f"Error adding batch {i} to {i+batch_size}: {e}")
#         break
# vectordb.persist()

# vectordb = FAISS.from_documents(chunks, embedding_model)
# FAISS.save_local(vectordb, "./faiss_db")

batch_size = 500  # safe batch size
vectordb = None
for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i+batch_size]
    if vectordb is None:
        vectordb = FAISS.from_documents(batch, embedding_model)
    else:
        vectordb.add_documents(batch)
    print(f"Processed chunks {i} to {i + len(batch)}")

FAISS.save_local(vectordb, "./faiss_db")
print("FAISS vector store saved successfully!")


c:\Users\Santosh\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Santosh\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Santosh\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or

AssertionError: Torch not compiled with CUDA enabled